## ✅ Import Libraries

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy torch

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from google.colab import drive


print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
!ls /content/Fatocheck/data/processed


In [ ]:
!git clone https://github.com/Susanta2025-lab/Fatocheck.git

In [ ]:
drive.mount('/content/drive')

## ✅ STEP 2 — Load Cleaned Dataset and Data Preparation

In [ ]:
# Load the cleaned news dataset
df = pd.read_csv("/content/drive/MyDrive/Fatocheck/cleaned_news.csv")

df.head()

In [ ]:
df.isnull().sum()

In [ ]:

df=df[["clean_content", "label"]].dropna()

In [ ]:
df.isnull().sum()

In [ ]:
df=df.rename(columns={"clean_content": "text"})

In [ ]:
df.head()

## ✅ STEP 3 — Train-Test-Validation Split

In [ ]:
# Split the dataset into training, validation, and test sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

## ✅ STEP 4 — Tokenizer + Dataset Conversion

In [ ]:
# Prepare the datasets for the transformer model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)


In [ ]:
# Tokenization function
def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding="max_length",
        truncation=True,
        max_length=256
        )

# Apply tokenization to the datasets
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

#rename the label column to labels
train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")

# Set the format for PyTorch
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

## ✅ STEP 5 — Load BERT

In [ ]:
# Load the pre-trained BERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
    )

## STEP 6 — Metrics

In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


## ✅ STEP 7 — Training Setup

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="models/trained/distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

## 🚀 STEP 8 — Train Model

In [ ]:
# Train the model
trainer.train()

## 💾 STEP 9 — Save model

In [ ]:
# Save the trained model and tokenizer

save_path = "/content/drive/MyDrive/Fatocheck/bert_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Saved at:", save_path)